# Regressione delta_skill_neve vs delta_skill_albedo

Confronto skill-vs-skill diretto tra due variabili con osservazione
indipendente vera (nessuna cover coinvolta, nessuna circolarita', nessun
disegno ibrido necessario): la stessa regione Russia-Cina-Siberia che mostra
una relazione delta_cover / delta_skill_tas (03) e delta_cover /
delta_skill_albedo (04) mostra, per osservazione diretta dei risultati, anche
una relazione tra `delta_skill_albedo` e `delta_skill_tas`. Questo notebook
verifica l'analoga relazione con la neve:

```
X = delta_skill_albedo  = skill_albedo_SENS - skill_albedo_CTRL (vs GLASS, genuino)
Y = delta_skill_neve     = skill_neve_SENS - skill_neve_CTRL (vs ERA5, genuino)
```

Nessuna variabile di cover ciclata (cvh/cvl): un solo confronto per
lead-year combo. Variabile di neve: `snd` di default, cambiare `snow_var`
sotto se serve passare a `sd`. Stesse tre mappe (Pearson slope, Pearson r,
Spearman rho) + scatter sul box Siberia di 03/04/05. Nessuna maschera a
varianza bassa qui: a differenza di `delta_cover`, `delta_skill_albedo` non
ha la stessa patologia strutturale (pixel a varianza quasi nulla per aree non
vegetate) — coerente col trattamento gia' usato per le variabili di skill
altrove in questa pipeline (mai mascherate).


In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
snow_var = 'snd'  # 'sd' dovrebbe essere equivalente, cambiare qui se serve
SAVE_PATH = str(FIG_DIR / "06_albedo_snow")  # sottocartella dedicata a questo notebook
os.makedirs(SAVE_PATH, exist_ok=True)


In [ ]:
# La logica di calcolo sta in cover_tas_lib.py (stesso modulo di 01/02/03/04/05,
# nuova funzione run_one_albedo_snow). Processi spawn freschi, stesso pattern
# robusto adottato in questa sessione.
sys.path.insert(0, os.getcwd())
from cover_tas_lib import run_one_albedo_snow, LEADS


In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, y1, y2, SAVE_PATH, snow_var) for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_albedo_snow, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
